# 00 - Data Preprocessing
**CSC61304 Group 6 - Machine Learning for Disaster Preparedness: Rainfall-Driven Flood Risk in Nepal**

Shared base notebook. Loads the CHIRPS rainfall dataset, cleans it, engineers
features, creates the flood risk label, and produces the train/test splits
used by all five models in this project.

Pipeline: **Load -> Clean -> Engineer Features -> Create Target Label ->
Handle Missing Values -> Encode -> Scale -> Split -> Save**


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

os.makedirs("../dataset/raw", exist_ok=True)
os.makedirs("../dataset/processed", exist_ok=True)

## Step 1-2: Load & Clean

Loads the raw CHIRPS CSV (row index 1 holds HXL tags and is skipped).
Filters to district-level records only, using the `adm_level` whose number
of distinct `PCODE`s matches Nepal's 77 districts.


In [ ]:
RAW_PATH = "../dataset/raw.csv"

df = pd.read_csv(RAW_PATH, skiprows=[1])

df.columns = [c.strip().lower() for c in df.columns]

if 'version' in df.columns:
    df = df.drop(columns=['version'])

df['date'] = pd.to_datetime(df['date'])

numeric_cols = ['rfh', 'r1h', 'r3h', 'rfh_avg', 'r1h_avg', 'r3h_avg',
                'rfq', 'r1q', 'r3q', 'n_pixels']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

level_counts = df.groupby('adm_level')['pcode'].nunique()
print("Unique pcodes per adm_level:\n", level_counts)

district_level = (level_counts - 77).abs().idxmin()
print(f"\nSelected adm_level = {district_level} as district level "
      f"({level_counts[district_level]} unique pcodes)")

df = df[df['adm_level'] == district_level].copy()
df = df.dropna(subset=['rfh']).drop_duplicates()

print(df.shape)
df.head()

## Step 3: Feature Engineering

Derives `year`, `month`, and `decade_num` (which 10-day period of the month)
from `date`. `district_zone` is derived from the province-code prefix of
`PCODE`, mapped to Terai/Hill/Mountain.


In [ ]:
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['decade_num'] = df['date'].dt.day.apply(lambda d: 1 if d <= 10 else (2 if d <= 20 else 3))

df['province_code'] = df['pcode'].astype(str).str[:4].str.upper()

province_zone_map = {
    'NP01': 'Hill',      # Koshi
    'NP02': 'Terai',     # Madhesh
    'NP03': 'Hill',      # Bagmati
    'NP04': 'Mountain',  # Gandaki
    'NP05': 'Terai',     # Lumbini
    'NP06': 'Mountain',  # Karnali
    'NP07': 'Hill',      # Sudurpashchim
}
df['district_zone'] = df['province_code'].map(province_zone_map)

print("Unmapped province codes (should be empty):")
print(df.loc[df['district_zone'].isna(), 'province_code'].unique())

df[['pcode', 'province_code', 'district_zone']].drop_duplicates().head(10)

## Step 4: Target Label Creation

Buckets `rfq` (rainfall anomaly %) into `flood_risk_level` using percentiles
of the observed distribution: Low, Moderate, High, and Extreme.


In [ ]:
q75, q90, q97 = df['rfq'].quantile([0.75, 0.90, 0.97])

def flood_risk_percentile(rfq):
    if rfq >= q97:
        return 'Extreme'
    elif rfq >= q90:
        return 'High'
    elif rfq >= q75:
        return 'Moderate'
    else:
        return 'Low'

df['flood_risk_level'] = df['rfq'].apply(flood_risk_percentile)

df['flood_risk_level'].value_counts(normalize=True)

## Step 5: Missing Value Handling

Fills missing numeric values using each district's own median (grouped by
`PCODE`), since rainfall patterns differ considerably between districts.


In [ ]:
for col in numeric_cols:
    df[col] = df.groupby('pcode')[col].transform(lambda x: x.fillna(x.median()))

df = df.dropna(subset=['rfq', 'flood_risk_level'])

df[numeric_cols].isna().sum()

## Step 6-7: Encoding + Scaling

Label-encodes `district_zone` and `flood_risk_level`, then builds two
feature sets:

- **Classification features** — used by Logistic Regression / Decision
  Tree, Naive Bayes, Random Forest, and K-Means.
- **Regression features** — used by Linear Regression, with its own
  separately-fit scaler.


In [ ]:
le_zone = LabelEncoder()
df['district_zone_enc'] = le_zone.fit_transform(df['district_zone'])

le_label = LabelEncoder()
df['flood_risk_level_enc'] = le_label.fit_transform(df['flood_risk_level'])

clf_feature_cols = ['rfh', 'r1h', 'r3h', 'rfh_avg', 'r1h_avg', 'r3h_avg',
                     'r1q', 'r3q', 'n_pixels',
                     'year', 'month', 'decade_num', 'district_zone_enc']
clf_feature_cols = [c for c in clf_feature_cols if c in df.columns]

clf_scaler = StandardScaler()
df_clf_scaled = df.copy()
df_clf_scaled[clf_feature_cols] = clf_scaler.fit_transform(df[clf_feature_cols])

reg_feature_cols = [c for c in clf_feature_cols if c != 'rfh']

reg_scaler = StandardScaler()
X_reg_scaled = df.copy()
X_reg_scaled[reg_feature_cols] = reg_scaler.fit_transform(df[reg_feature_cols])

df_clf_scaled[clf_feature_cols].describe()

## Step 8: Train/Test Split + Save

Classification split is stratified on `flood_risk_level_enc`. Regression
target (`y_reg`) is kept in raw mm so RMSE/MAE report in interpretable
rainfall units.


In [ ]:
X = df_clf_scaled[clf_feature_cols]
y_class = df_clf_scaled['flood_risk_level_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

X_reg = X_reg_scaled[reg_feature_cols]
y_reg = df['rfh']

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

df.to_csv("../dataset/processed/nepal_flood_processed.csv", index=False)

X_train.to_csv("../dataset/processed/X_train.csv", index=False)
X_test.to_csv("../dataset/processed/X_test.csv", index=False)
y_train.to_csv("../dataset/processed/y_train.csv", index=False)
y_test.to_csv("../dataset/processed/y_test.csv", index=False)

X_train_reg.to_csv("../dataset/processed/X_train_reg.csv", index=False)
X_test_reg.to_csv("../dataset/processed/X_test_reg.csv", index=False)
y_train_reg.to_csv("../dataset/processed/y_train_reg.csv", index=False)
y_test_reg.to_csv("../dataset/processed/y_test_reg.csv", index=False)

print("Saved processed data and both splits to dataset/processed/")
print(f"\nClassification features ({len(clf_feature_cols)}): {clf_feature_cols}")
print(f"Regression features ({len(reg_feature_cols)}): {reg_feature_cols}")